# Generate figures for the empirical assessment on P1000 matched somatic +/- germline data
Here, we analyze results of running models on empirical data: matched somatic +/- germline data from the P1000 dataset.

Prerequisites: 
- you ran the empirical assessment experiment over some number of independent, stratified train/val/test splits
- the results are saved as a CSV (you can generate this by running the relevant portion of the `notebooks/analysis/make_supplementary_tables.ipynb` notebook)

In [ ]:
# Import Required Libraries
import logging
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from pnet.data_analysis import permutation_utils as pu
from pnet.data_analysis import plotting_utils as plot_utils
from pnet.data_analysis import importance_utils as import_utils

sys.path.insert(0, "../..")  # add project_config to path
import project_config

# Setup Logging and Configuration
logging.basicConfig(
    format="%(asctime)s %(levelname)-8s [%(name)s] %(message)s",
    level=logging.INFO,
    datefmt="%Y-%m-%d %H:%M:%S",
    force=True,
)
logger = logging.getLogger(__name__)

%load_ext autoreload
%autoreload 2

## Hyperparameter definition (inputs / outputs / figure details)

In [ ]:
# Path to file with performance metrics and feature importance paths for each run
METRIC_CSV_PATH = (
    project_config.SUPPLEMENTARY_TABLES_DIR
    / "p1000_predictionMetrics_per_datasplit_full.csv"
)

# Define directories for saving results
SAVE_RESULTS_NAME = "datasplit_analysis_p1000"
SAVE_FIGS_DIR = project_config.FIGURE_3_DIR / SAVE_RESULTS_NAME
SAVE_TABLES_DIR = project_config.SUPPLEMENTARY_TABLES_DIR #/ SAVE_RESULTS_NAME

# make directory if it doesn't exist
os.makedirs(SAVE_FIGS_DIR, exist_ok=True)
os.makedirs(SAVE_TABLES_DIR, exist_ok=True)

# Figure parameters
figsize_tuple = (2.1, 2)
aspect_ratio_value = 1.7
cmap_style = "colorblind"

# germline lighter version
germline_lighter_colorblind_split = {
    "somatic_amp somatic_del somatic_mut germline_rare_lof_missense": "#DE8F05",
    "somatic_amp somatic_del somatic_mut germline_rare_missense": "#029E73", # renamed: germline_rare_common_missense --> germline_rare_missense
    "somatic_amp somatic_del somatic_mut germline_rare_lof": "#CC78BC",
    "somatic_amp somatic_del somatic_mut": "#949494",
    "germline_rare_lof_missense": "#f1a934",
    "germline_rare_missense": "#02be8a", # renamed: germline_rare_common_missense --> germline_rare_missense
    "germline_rare_lof": "#dea7d3",
}
custom_colors = germline_lighter_colorblind_split

model_type_name_map = {
    "pnet": "P-NET",
    "rf": "Random Forest",
    "logistic_regression": "Logistic Regression",
}
col_order = ["P-NET", "Random Forest", "Logistic Regression"]
model_color_map = {
    "pnet": "#1f77b4",  # blue
    "rf": "#ff7f0e",  # orange
    "logistic_regression": "#2ca02c",  # green
}

pair_col = "split_id"  # your pairing column for the wilcoxon test (same-splits)

eval_set = "test"  # "test" or "train". Run through once for each to generate all the figure panels for P1000 results.

## Load performance metrics
Loading performance metrics, which can be grouped by `model_type`, `datasets`

In [ ]:
df = pd.read_csv(METRIC_CSV_PATH)


print(df.datasets.value_counts())

# Average over replicates
group_by_cols = ["model_type", "datasets"]

# Add a counts column for each group
df_counts = df.groupby(group_by_cols).size().reset_index(name="count")

# Average over seeds
df_avg = df.groupby(group_by_cols, as_index=False).mean(numeric_only=True)

# Merge counts into df_avg
df_avg = df_avg.merge(df_counts, on=group_by_cols)
df_avg

In [ ]:
# Define the prevalence of the positive class (equal to random model's AP or AUPRC)
p1000_metastatic_prevalence = pd.read_csv(
    os.path.join(df.input_data_dir[0], "y.csv"), index_col=0
)["is_met"].mean()
print(f"Prevalence of 1s (metastatic samples): {p1000_metastatic_prevalence:.2%}")

In [ ]:
df.datasets.value_counts()

# Statistical comparisons at replicate level (across data splits): model comparison, and dataset comparisons
We will use Wilcoxon signed-rank test to determine if there are any pairwise performance differences between
- P-NET vs. RF, P-NET vs. LR
- a reference germline-only dataset (either germline_rare_lof, most sparse; or germline_rare_common_lof_missense, least sparse) vs. other germline-only dataset
- somatic-only dataset vs. somatic + germline datasets

## Does P-NET outperform either RF or LR?
Going to use a two-sided Wilcoxon signed-rank test since this is nonparametric (needed since have small n replicates) and works with paired data. Will test this separately on each dataset.

In [ ]:
metric = f"{eval_set}_avg_precision"

model_pairs = [
    ("pnet", "rf"),
    ("pnet", "logistic_regression"),
    ("rf", "logistic_regression"),
]

df_model_comp_ext = pu.compute_paired_model_wilcoxon_results(
    df_runs=df,
    metric=metric,
    replicate_col="split_id",
    model_pairs=model_pairs,
    fdr_correction=True,   # optional
    fdr_groupby_cols=None, # global
)

df_model_comp_ext.to_csv(
    SAVE_TABLES_DIR / f"p1000_datasplit_modelComparison_{eval_set}Set.csv",
    float_format="%.3f",
    index=False,
)

display(df_model_comp_ext.round(3).head())

In [ ]:
df_model_comp_ext.columns.tolist()

## Statistical significance of delta performance across different input datasets

In [ ]:
metric = f"{eval_set}_avg_precision"

dataset_pairs = pu.make_dataset_pairs_baseline_vs_many(df["datasets"].unique(), 
                                                       germline_baseline="germline_rare_lof", 
                                                       somatic_baseline="somatic_amp somatic_del somatic_mut")
logging.info(f"Dataset pairs (n={len(dataset_pairs)}) for Wilcoxon test: \n{dataset_pairs} ")

df_dset_comp = pu.compute_paired_dataset_wilcoxon_results(
    df_runs=df,
    dataset_pairs=dataset_pairs,
    metric=metric,
    replicate_col="split_id",
    group_cols=["model_type"],
    fdr_groupby_cols=["model_type"],
    fdr_correction=True,
    require_matching_replicates=True,
)

df_dset_comp.to_csv(
    SAVE_TABLES_DIR / f"p1000_datasplit_datasetComparison_{eval_set}Set.csv",
    float_format="%.3f",
    index=False,
)

display(df_dset_comp.round(3))

## helper functions

In [ ]:
def smart_ticklabels(labels):
    return [f"{int(l)}" if float(l).is_integer() else f"{l:.1f}" for l in labels]


def get_smart_odds_ratio_labels(axessubplot_object):
    """Expects type <class 'matplotlib.axes._subplots.AxesSubplot'> as input"""
    smart_yticks = smart_ticklabels(
        [float(l.get_text()) for l in axessubplot_object.get_yticklabels()]
    )
    return smart_yticks


def generate_fake_data(datasets):
    np.random.seed(123)
    data = []
    for i, dataset in enumerate(datasets):
        for _ in range(20):
            value = np.random.normal(loc=5 + i, scale=1.0)
            data.append({"datasets": dataset, "value": value})
    return pd.DataFrame(data)


def plot_boxplot(
    data,
    x_name="datasets",
    y_name="value",
    color_map=None,
    ax=None,
    no_x_labels=False,
    showfliers=True,
    dataset_order=None,
):
    """
    Plot a seaborn boxplot with optional custom color mapping.
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 5))

    valid_order = (
        get_valid_group_order(data, dataset_order, group_col=x_name)
        if dataset_order
        else None
    )
    sns.boxplot(
        data=data,
        x=x_name,
        y=y_name,
        ax=ax,
        showfliers=showfliers,
        palette=color_map if color_map else None,
        order=valid_order,
    )
    if not showfliers:
        sns.stripplot(
            data=data,
            x=x_name,
            y=y_name,
            ax=ax,
            color="black",
            jitter=0.2,
            alpha=0.3,
            order=valid_order,
        )
    if no_x_labels:
        ax.tick_params(axis="x", labelbottom=False)
    else:
        ax.tick_params(axis="x", labelbottom=True)
        plt.setp(ax.get_xticklabels(), rotation=90)
    return ax


def plot_boxplot_with_features(
    data,
    datasets,
    feature_matrix,
    color_map=None,
    dataset_order=None,
    x_name="datasets",
    y_name="value",
    title="Boxplot with UpSet-Style Feature Labels",
    figsize_tuple=(6, 3),
    fig_ratio=[4, 1],
):
    # valid data order: restrict the ordered list to the groups we actually have in our data; default to datasets
    valid_dataset_order = (
        get_valid_group_order(data, dataset_order, group_col=x_name)
        if dataset_order
        else datasets
    )

    fig, (ax_box, ax_matrix) = plt.subplots(
        2,
        1,
        figsize=figsize_tuple,
        gridspec_kw={"height_ratios": fig_ratio},
        sharex=True,
    )

    plot_boxplot(
        data,
        x_name,
        y_name,
        color_map=color_map,
        ax=ax_box,
        no_x_labels=True,
        dataset_order=valid_dataset_order,
    )
    ax_box.set_title(title)

    ax_matrix = plot_utils.plot_feature_matrix(
        feature_matrix, x_labels=valid_dataset_order, ax=ax_matrix
    )

    for i in range(len(valid_dataset_order)):
        ax_box.axvline(x=i, color="lightgray", linestyle="--", linewidth=1, zorder=0)
        ax_matrix.axvline(x=i, color="lightgray", linestyle="--", linewidth=1, zorder=0)

    plt.tight_layout()
    return plt


def get_group_ordered_by_statistic(
    df, group_col_name, order_col_name="auc", stat="median", ascending=False
):
    if stat == "mean":
        # Calculate mean for each dataset
        sorted_df = (
            df.groupby(group_col_name)[order_col_name]
            .mean()
            .sort_values(ascending=ascending)
        )
    elif stat == "median":
        # Calculate median for each dataset
        sorted_df = (
            df.groupby(group_col_name)[order_col_name]
            .median()
            .sort_values(ascending=ascending)
        )
    else:
        print("TODO: not implemented")
        return
    # Order the datasets by descending mean AUC
    group_order = sorted_df.index.tolist()
    return group_order


def get_valid_group_order(df, group_order, group_col="datasets"):
    """
    Return a filtered dataset order that includes only the group_col values present in df.
    """
    available = df[group_col].unique().tolist()
    return [d for d in group_order if d in available]


def plot_model_comparison_with_features(
    df,
    eval_set,
    feature_matrix,
    color_map=None,
    dataset_order=None,
    figsize_tuple=(6, 8),
    fig_ratio=[6, 6, 6, 3],
    shareY=True,
    title="Performance boxplots",
    random_performance_value=None,
    setMinYToRandomPerformance=False,
):
    metric_col = f"{eval_set}_avg_precision"
    datasets = df["datasets"].unique().tolist()
    valid_dataset_order = (
        get_valid_group_order(df, dataset_order, group_col="datasets")
        if dataset_order
        else datasets
    )

    # Split data
    df_pnet = df[df["model_type"] == "pnet"]
    df_rf = df[df["model_type"] == "rf"]
    df_lr = df[df["model_type"] == "logistic_regression"]

    # Set up figure
    fig, (ax_pnet, ax_rf, ax_lr, ax_matrix) = plt.subplots(
        4,
        1,
        figsize=figsize_tuple,
        gridspec_kw={"height_ratios": fig_ratio},
        sharex=True,
    )

    # Plot PNet
    plot_boxplot(
        data=df_pnet,
        x_name="datasets",
        y_name=metric_col,
        color_map=color_map,
        ax=ax_pnet,
        dataset_order=valid_dataset_order,
    )
    ax_pnet.set_title("model = P-NET")

    # Plot RF
    plot_boxplot(
        data=df_rf,
        x_name="datasets",
        y_name=metric_col,
        color_map=color_map,
        ax=ax_rf,
        dataset_order=valid_dataset_order,
    )
    ax_rf.set_title("model = Random Forest")

    # Plot LR
    plot_boxplot(
        data=df_lr,
        x_name="datasets",
        y_name=metric_col,
        color_map=color_map,
        ax=ax_lr,
        dataset_order=valid_dataset_order,
    )
    ax_lr.set_title("model = Logistic Regression")

    if shareY:
        # Sync y-axis limits between ax_pnet and ax_rf
        y_min = min(ax_pnet.get_ylim()[0], ax_rf.get_ylim()[0], ax_lr.get_ylim()[0])
        if setMinYToRandomPerformance and random_performance_value is not None:
            y_min = min(
                y_min, random_performance_value - 0.03
            )  # add a little buffer below the random performance line
        y_max = max(ax_pnet.get_ylim()[1], ax_rf.get_ylim()[1], ax_lr.get_ylim()[1])
        ax_pnet.set_ylim(y_min, y_max)
        ax_rf.set_ylim(y_min, y_max)
        ax_lr.set_ylim(y_min, y_max)

    # add horizontal line to represent random performance
    if random_performance_value is not None:
        ax_pnet.axhline(
            y=random_performance_value, linestyle="--", color="black", linewidth=1
        )
        ax_rf.axhline(
            y=random_performance_value, linestyle="--", color="black", linewidth=1
        )
        ax_lr.axhline(
            y=random_performance_value, linestyle="--", color="black", linewidth=1
        )

    # Draw vertical lines for each dataset to draw the eye to the x-axis legend
    for i in range(len(datasets)):
        ax_pnet.axvline(x=i, color="lightgray", linestyle="--", linewidth=1, zorder=0)
        ax_rf.axvline(x=i, color="lightgray", linestyle="--", linewidth=1, zorder=0)
        ax_lr.axvline(x=i, color="lightgray", linestyle="--", linewidth=1, zorder=0)
        ax_matrix.axvline(x=i, color="lightgray", linestyle="--", linewidth=1, zorder=0)

    # Plot shared feature matrix
    plot_utils.plot_feature_matrix(feature_matrix, x_labels=valid_dataset_order, ax=ax_matrix)
    plt.suptitle(title)
    plt.tight_layout()
    return fig


def plot_model_comparison_with_features_grouped(
    df,
    eval_set,
    feature_matrix,
    color_map=None,  # maps dataset -> color (for boxes)
    dataset_order=None,
    figsize_tuple=(9, 5.5),
    fig_ratio=(6, 3),
    shareY=True,
    title="Performance boxplots",
    random_performance_value=None,
    setMinYToRandomPerformance=False,
    model_order=("pnet", "rf", "logistic_regression"),
    model_labels=None,  # optional pretty labels
    box_width=0.22,
    group_gap=0.08,
):
    """
    Grouped boxplot: for each dataset, show side-by-side boxes for models (pnet/rf/lr),
    plus a feature matrix panel below.

    Requirements:
      - df columns: ["datasets", "model_type", f"{eval_set}_avg_precision"]
      - feature_matrix compatible with your existing plot_feature_matrix(...)
      - get_valid_group_order(...) exists (as in your code)

    Example:
        p3_grouped = plot_model_comparison_with_features_grouped(
            df_germline_only,
            eval_set,
            feature_matrix,
            random_performance_value=p1000_metastatic_prevalence,
            setMinYToRandomPerformance=True,
            color_map=custom_colors,
            model_labels=model_type_name_map,
            dataset_order=dataset_order,
            figsize_tuple=(8, 3),
            fig_ratio=(6, 3),
            title=f"AUPRC on germline datasets ({eval_set} set, n=10)",
        )
    """
    metric_col = f"{eval_set}_avg_precision"

    datasets = df["datasets"].unique().tolist()
    valid_dataset_order = (
        get_valid_group_order(df, dataset_order, group_col="datasets")
        if dataset_order
        else datasets
    )

    # --- figure layout (top: grouped boxplot, bottom: feature matrix)
    fig, (ax, ax_matrix) = plt.subplots(
        2,
        1,
        figsize=figsize_tuple,
        gridspec_kw={"height_ratios": list(fig_ratio)},
        sharex=True,
    )

    # --- compute x positions: group per dataset, offsets per model
    n_d = len(valid_dataset_order)
    n_m = len(model_order)

    group_centers = np.arange(n_d)
    # offsets centered around 0, e.g. for 3 models -> [-w, 0, +w]
    offsets = (np.arange(n_m) - (n_m - 1) / 2.0) * (box_width + group_gap)

    # --- draw grouped boxplots
    all_ymins, all_ymaxs = [], []

    for mi, model in enumerate(model_order):
        x_positions = group_centers + offsets[mi]

        data_for_boxes = []
        for ds in valid_dataset_order:
            vals = (
                df.loc[
                    (df["datasets"] == ds) & (df["model_type"] == model),
                    metric_col,
                ]
                .dropna()
                .values
            )
            data_for_boxes.append(vals)

        bp = ax.boxplot(
            data_for_boxes,
            positions=x_positions,
            widths=box_width,
            patch_artist=True,
            manage_ticks=False,
            showfliers=(
                model != "logistic_regression"
            ),  # hide fliers for LR (usually very low)
            label=model_labels.get(model, model) if model_labels else model,
            medianprops=dict(color="grey", linewidth=1.5),
        )

        # style boxes: dataset-based colors (consistent with your previous approach)
        for box in bp["boxes"]:
            box.set_facecolor(color_map.get(model, "lightgray"))
            box.set_alpha(0.9)

        # optional: make medians stand out a bit
        for med in bp["medians"]:
            med.set_linewidth(1.2)

        # gather y-lims from drawn artists (use whiskers/extrema)
        # (matplotlib doesn't directly return global y-range; just grab current and merge later)
        all_ymins.append(ax.get_ylim()[0])
        all_ymaxs.append(ax.get_ylim()[1])

    # --- x ticks at dataset centers
    ax.set_xticks(group_centers)
    ax.set_xticklabels(valid_dataset_order, rotation=90)

    ax.set_ylabel(metric_col)
    ax.set_title(title)

    # --- shared y-limits + random baseline line
    if shareY:
        y_min = min(all_ymins) if all_ymins else ax.get_ylim()[0]
        y_max = max(all_ymaxs) if all_ymaxs else ax.get_ylim()[1]
        if setMinYToRandomPerformance and random_performance_value is not None:
            y_min = min(y_min, random_performance_value - 0.03)
        ax.set_ylim(y_min, y_max)

    random_line = None
    if random_performance_value is not None:
        random_line = ax.axhline(
            y=random_performance_value,
            linestyle="--",
            color="grey",
            linewidth=1,
            zorder=1,
            label="Random",
        )

    # --- feature matrix panel
    # IMPORTANT: x_labels must match dataset order (centers are 0..n_d-1)
    plot_utils.plot_feature_matrix(feature_matrix, x_labels=valid_dataset_order, ax=ax_matrix)

    # alternating vertical background bars (axvspan)
    for i in range(len(valid_dataset_order)):
        if i % 2 == 0:  # alternate
            ax.axvspan(
                i - 0.5,
                i + 0.5,
                color="lightgrey",
                alpha=0.2,
                zorder=0,
            )
            ax_matrix.axvspan(
                i - 0.5,
                i + 0.5,
                color="lightgrey",
                alpha=0.2,
                zorder=0,
            )

    # --- legend for model, offset
    ax.legend(
        title="Model",
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0),
        frameon=True,
    )

    ax.grid(
        axis="y",
        linestyle="-",
        linewidth=0.8,
        alpha=0.2,
        zorder=0,
    )
    ax.set_axisbelow(True)

    plt.tight_layout()
    return fig


## Model comparison: AUPRC plots with and without statistical annotations

In [ ]:
# from pnet.data_analysis.plotting_utils import plot_grouped_stripplot_with_medians_and_feature_matrix
metric_col = f"{eval_set}_avg_precision"
df_pnet = df[df["model_type"] == "pnet"]
datasets = df["datasets"].unique().tolist()
height_ratios = (6, 1.8)
somatic_figsize_tuple = (6,3)
germline_figsize_tuple = (5,3)

# force datasets to follow this order (decreasing median(metric) in P-NET runs)
dataset_order = get_group_ordered_by_statistic(
    df_pnet,
    group_col_name="datasets",
    stat="median",
    order_col_name=metric_col,
    ascending=False,
)

# actually, going to manually set the order so that easier to make comparisons:
# change dataset order for easier dataset-dataset comparison between somatic+germline and germline-only panels
dataset_order = [
'somatic_amp somatic_del somatic_mut',
'somatic_amp somatic_del somatic_mut germline_rare_lof_missense',
'somatic_amp somatic_del somatic_mut germline_rare_lof',
'somatic_amp somatic_del somatic_mut germline_rare_missense',
'germline_rare_lof_missense',
'germline_rare_lof',
'germline_rare_missense',
]

metric_col = f"{eval_set}_avg_precision"
pair_col = "split_id"  # your pairing column for the wilcoxon test (same-splits)
model_order = ["pnet", "rf", "logistic_regression"]
model_labels = {
    "pnet": "P-NET",
    "rf": "Random Forest",
    "logistic_regression": "Logistic Regression",
}
# model colors (palette)
model_color_map = {"pnet": "#1f77b4", "rf": "#ff7f0e", "logistic_regression": "#2ca02c"}
random_performance_value = p1000_metastatic_prevalence

logging.info(f"Performance of germline only with annotations on the {eval_set}")
df_germline_only = df[~df["datasets"].str.contains("somatic")]
feature_matrix = plot_utils.get_feature_matrix_hierarchical(datasets=df_germline_only["datasets"].unique().tolist())

fig, (ax_main, ax_matrix), annotator = plot_utils.plot_grouped_stripplot_with_medians_and_feature_matrix(
    df=df_germline_only,
    metric_col=f"{eval_set}_avg_precision",
    pair_col=pair_col,
    model_order=["pnet", "rf", "logistic_regression"],
    model_labels=model_type_name_map,
    model_color_map=model_color_map,
    random_performance_value=p1000_metastatic_prevalence,
    dataset_order=dataset_order,
    feature_matrix=feature_matrix,
    plot_feature_matrix_func=plot_utils.plot_feature_matrix,     # pass existing function
    df_model_comp=df_model_comp_ext,                      # optional
    title=f"AUPRC on germline datasets ({eval_set} set, n=10)",
    strip_size=4,
    height_ratios=(6,1.55),
    figsize=(5, 3),
    y_label=f"{eval_set} AUPRC",
    draw_legend=False,
    annot_style="stars",
    annot_no_sig_name=True,
)
# plt.tight_layout()
plt.savefig(
    os.path.join(
        SAVE_FIGS_DIR, f"datasplit_P1000_grouped_model_comparison_auprc_germline_annotated_{eval_set}_set.png"
    ),
    format="png",
    dpi=600,
    bbox_inches="tight",
)
plt.show()

logging.info(f"Performance of germline only without annotations on the {eval_set}")
fig, (ax_main, ax_matrix), annotator = plot_utils.plot_grouped_stripplot_with_medians_and_feature_matrix(
    df=df_germline_only,
    metric_col=f"{eval_set}_avg_precision",
    pair_col=pair_col,
    model_order=["pnet", "rf", "logistic_regression"],
    model_labels=model_type_name_map,
    model_color_map=model_color_map,
    random_performance_value=p1000_metastatic_prevalence,
    dataset_order=dataset_order,
    feature_matrix=feature_matrix,
    plot_feature_matrix_func=plot_utils.plot_feature_matrix,
    # df_model_comp=df_model_comp_ext,                      # optional
    title=f"AUPRC on germline datasets ({eval_set} set, n=10)",
    strip_size=4,
    height_ratios=(6,1.55),
    figsize=(5, 3),
    y_label=f"{eval_set} AUPRC",
    draw_legend=False,
)
# plt.tight_layout()
plt.savefig(
    os.path.join(
        SAVE_FIGS_DIR, f"datasplit_P1000_grouped_model_comparison_auprc_germline_{eval_set}_set.png"
    ),
    format="png",
    dpi=600,
    bbox_inches="tight",
)
plt.show()

# somatic datasets
logging.info(f"Performance of somatic containing datasets with annotations on the {eval_set}")
df_combos = df[df["datasets"].str.contains("somatic")]
feature_matrix = plot_utils.get_feature_matrix_hierarchical(datasets=df_combos["datasets"].unique().tolist())


fig, (ax_main, ax_matrix), annotator = plot_utils.plot_grouped_stripplot_with_medians_and_feature_matrix(
    df=df_combos,
    metric_col=f"{eval_set}_avg_precision",
    pair_col=pair_col,
    model_order=["pnet", "rf", "logistic_regression"],
    model_labels=model_type_name_map,
    model_color_map=model_color_map,
    dataset_order=dataset_order,
    feature_matrix=feature_matrix,
    plot_feature_matrix_func=plot_utils.plot_feature_matrix,
    df_model_comp=df_model_comp_ext,
    title=f"AUPRC on somatic +/- germline datasets ({eval_set} set, n=10)",
    strip_size=4,
    height_ratios=(6,1.55),
    figsize=(5.5, 3),
    y_label=f"{eval_set} AUPRC",
    draw_legend=False,
    annot_style="stars",
    annot_no_sig_name=True,
)
# plt.tight_layout()
plt.savefig(
    os.path.join(
        SAVE_FIGS_DIR, f"datasplit_P1000_grouped_model_comparison_auprc_somatic_germline_annotated_{eval_set}_set.png"
    ),
    format="png",
    dpi=600,
    bbox_inches="tight",

)
plt.show()

logging.info(f"Performance of somatic containing datasets without annotations on the {eval_set}")
fig, (ax_main, ax_matrix), annotator = plot_utils.plot_grouped_stripplot_with_medians_and_feature_matrix(
    df=df_combos,
    metric_col=f"{eval_set}_avg_precision",
    pair_col=pair_col,
    model_order=["pnet", "rf", "logistic_regression"],
    model_labels=model_type_name_map,
    model_color_map=model_color_map,
    dataset_order=dataset_order,
    feature_matrix=feature_matrix,
    plot_feature_matrix_func=plot_utils.plot_feature_matrix,
    # df_model_comp=df_model_comp_ext,
    title=f"AUPRC on somatic +/- germline datasets ({eval_set} set, n=10)",
    strip_size=4,
    height_ratios=(6,1.55),
    figsize=(5.5, 3),
    y_label=f"{eval_set} AUPRC",
    draw_legend=False,
)
# plt.tight_layout()
plt.savefig(
    os.path.join(
        SAVE_FIGS_DIR, f"datasplit_P1000_grouped_model_comparison_auprc_somatic_germline_{eval_set}_set.png"
    ),
    format="png",
    bbox_inches="tight",
    dpi=600,
)
plt.show()


In [ ]:
# save legend separately for reuse in other figures
logging.info("Saving legend-only figure for model comparison plot")
fig, _, _ = plot_utils.plot_grouped_stripplot_with_medians_and_feature_matrix(
    df=df,
    metric_col=metric_col,
    pair_col=pair_col,
    model_order=model_order,
    model_labels=model_labels,
    model_color_map=model_color_map,
    dataset_order=dataset_order,

    # legend-only behavior
    save_legend_path=os.path.join(SAVE_FIGS_DIR, "P1000_grouped_model_comparison_legend.png"),
    draw_legend=False,

    # everything else can be minimal / dummy
    feature_matrix=None,
    df_model_comp=None,
    title=None,
)
plt.close(fig)


In [ ]:
# TODO: visualizing all three model-model comparisons on the same plot. Not currently including in the manuscript, but nice to see.
# somatic datasets
logging.info(
    f"Performance of somatic containing datasets with annotations on the {eval_set}"
)
df_combos = df[df["datasets"].str.contains("somatic")]
feature_matrix = plot_utils.get_feature_matrix_hierarchical(
    datasets=df_combos["datasets"].unique().tolist()
)


fig, (ax_main, ax_matrix), annotator = (
    plot_utils.plot_grouped_stripplot_with_medians_and_feature_matrix(
        df=df_combos,
        comparisons=[
            ("pnet", "rf"),
            ("pnet", "logistic_regression"),
            ("rf", "logistic_regression"),
        ],
        metric_col=f"{eval_set}_avg_precision",
        pair_col=pair_col,
        model_order=["pnet", "rf", "logistic_regression"],
        model_labels=model_type_name_map,
        model_color_map=model_color_map,
        dataset_order=dataset_order,
        feature_matrix=feature_matrix,
        plot_feature_matrix_func=plot_utils.plot_feature_matrix,
        df_model_comp=df_model_comp_ext,
        title=f"AUPRC on somatic +/- germline datasets ({eval_set} set, n=10)",
        strip_size=4,
        height_ratios=(6, 1.55),
        figsize=(5.5, 3),
        y_label=f"{eval_set} AUPRC",
        draw_legend=False,
        annot_style="stars",
        annot_no_sig_name=True,
    )
)


logging.info(f"Performance of germline only with annotations on the {eval_set}")
df_germline_only = df[~df["datasets"].str.contains("somatic")]
feature_matrix = plot_utils.get_feature_matrix_hierarchical(datasets=df_germline_only["datasets"].unique().tolist())

fig, (ax_main, ax_matrix), annotator = (
    plot_utils.plot_grouped_stripplot_with_medians_and_feature_matrix(
        df=df_germline_only,
        comparisons=[
            ("pnet", "rf"),
            ("pnet", "logistic_regression"),
            ("rf", "logistic_regression"),
        ],
        metric_col=f"{eval_set}_avg_precision",
        pair_col=pair_col,
        model_order=["pnet", "rf", "logistic_regression"],
        model_labels=model_type_name_map,
        model_color_map=model_color_map,
        dataset_order=dataset_order,
        feature_matrix=feature_matrix,
        plot_feature_matrix_func=plot_utils.plot_feature_matrix,
        df_model_comp=df_model_comp_ext,
        title=f"AUPRC on germline datasets ({eval_set} set, n=10)",
        strip_size=4,
        height_ratios=(6, 1.55),
        figsize=(5.5, 3),
        y_label=f"{eval_set} AUPRC",
        draw_legend=False,
        annot_style="stars",
        annot_no_sig_name=True,
    )
)

## Performance boxplot of P-NET and RF

### boxplots: germline only

In [ ]:
metric_col = f"{eval_set}_avg_precision"
df_germline_only = df[~df["datasets"].str.contains("somatic")]
datasets = df_germline_only["datasets"].unique().tolist()


feature_matrix = plot_utils.get_feature_matrix_hierarchical(datasets)

p3 = plot_utils.plot_model_comparison_with_features(
    df_germline_only, eval_set, feature_matrix,
    color_map=custom_colors,
    dataset_order=dataset_order,
    random_performance_value=p1000_metastatic_prevalence,
    setMinYToRandomPerformance=True,
    figsize_tuple=(5, 7),
    fig_ratio=[5, 5, 5, 1.5],
    title=f"AUPRC on germline datasets ({eval_set} set, n=10)",
    show_boxplot=False,
    show_stripplot=True,
    show_medians=True,
    add_vertical_guides=False,
    strip_size=4,
    strip_edge_alpha=0.9,
    strip_face_alpha=0.4,
    df_dset_comp=df_dset_comp,
    y_label=f"{eval_set} AUPRC",
    annot_style="stars",
    annot_no_sig_name=True,
)

p3.savefig(
    os.path.join(
        SAVE_FIGS_DIR, f"datasplit_P1000_auprc_germline_only_model_comparison_annotated_{eval_set}_set.png"
    ),
    format="png",
    dpi=600,
)
p3.show()



### boxplot: somatic:germline combos

In [ ]:
metric_col = f"{eval_set}_avg_precision"
df_combos = df[df["datasets"].str.contains("somatic")]
datasets = df_combos["datasets"].unique().tolist()

feature_matrix = plot_utils.get_feature_matrix_hierarchical(datasets)

# with annotations
p4_annot = plot_utils.plot_model_comparison_with_features(
    df_combos, eval_set, feature_matrix,
    color_map=custom_colors,
    dataset_order=dataset_order,
    setMinYToRandomPerformance=True,
    figsize_tuple=(5, 7),
    fig_ratio=[5, 5, 5, 1.5],
    title=f"AUPRC on somatic +/- germline datasets ({eval_set} set, n=10)",
    show_boxplot=False,
    show_stripplot=True,
    show_medians=True,
    add_vertical_guides=False,
    strip_size=4,
    strip_edge_alpha=0.9,
    strip_face_alpha=0.4,
    df_dset_comp=df_dset_comp,
    y_label=f"{eval_set} AUPRC",
    annot_style="stars",
    annot_no_sig_name=True,

)

p4_annot.savefig(
    os.path.join(
        SAVE_FIGS_DIR, f"datasplit_P1000_auprc_somatic_germline_model_comparison_annotated_{eval_set}_set.png"
    ),
    format="png",
    dpi=600,
)
p4_annot.show()

# Feature importance

In [ ]:
logger.info("Get the feature importances and ranks for each dataset combination")
eval_set = "test"
feature_importance_path = (
    f"{eval_set}_feature_importances_path"
)
importance_path = (
    f"{eval_set}_gene_importances_path"  # f'{eval_set}_gene_importances_path
)
logger.info("Filter to runs where model_type is 'pnet'")
df_filt = df[df["model_type"] == "pnet"]
response_f = "/mnt/disks/gmiller_data1/pnet_germline/data/pnet_database/prostate/processed/response_paper.csv"
response_df = import_utils.load_response_variable(response_f)

In [ ]:
# constants used below
IMPORTANCE_PATH_COL = importance_path  # must exist in df_filt
GROUP_IDENTIFIER_COL = "datasets"
REPLICATE_COL = "split_id"
GENE_NAME = "BRCA2"
DATASET_COL = "datasets"
REPLICATE_COL_OUT = "split_id"
METRIC_COL = "rank"  # do not flip sign; lower = better


### BRCA2 rank: Wilcoxon tests (model comparison, dataset comparison)

In [ ]:
# Single chunk: build per-run ranks and run paired Wilcoxon on BRCA2 rank

# --- User-configurable inputs (edit as needed) --------------------------------
# df_filt                  # your dataframe of feature_importance paths / metadata
# response_df              # response DataFrame indexed by sample id
# importance_path          # column name in df_filt containing the path to imps files
# dataset_pairs            # list of (dataset_A, dataset_B) tuples to compare
# --------------------------------------------------------------------------------

# constructing the dataset pairs of interest
dataset_pairs = pu.make_dataset_pairs_baseline_vs_many(df["datasets"].unique(), 
                                                       germline_baseline="germline_rare_lof_missense", 
                                                       somatic_baseline="somatic_amp somatic_del somatic_mut")
dataset_pairs += [('germline_rare_lof_missense', 'somatic_amp somatic_del somatic_mut')]
logging.info(f"Dataset pairs (n={len(dataset_pairs)}) for Wilcoxon test: \n{dataset_pairs} ")

logger.info("Starting BRCA2 paired-rank analysis...")

# 1) Build per-run importances and ranks (will raise if replicate col missing or duplicates)
logger.info("Processing per-run feature importances and ranks (indexing by replicate)...")
df_imps_by_key, df_ranks_by_key = import_utils.process_feature_importances(
    df_feature_importance_paths=df_filt,
    response_df=response_df,
    importance_path_column=IMPORTANCE_PATH_COL,
    group_identifier_column=GROUP_IDENTIFIER_COL,
    replicate_col=REPLICATE_COL,
)

# quick check that we produced at least one key
if len(df_ranks_by_key) == 0:
    raise RuntimeError("No keys produced by process_feature_importances(). Check your paths and input frames.")
for dataset, _ in df_imps_by_key.items():
    logger.debug(f"Index of each rank DF (df_ranks_by_key[dataset].index) should be random replicates: {df_ranks_by_key[dataset].index}")

logger.info(f"Processed feature importances for {len(df_ranks_by_key)} dataset keys.")

# 2) Build per-run BRCA2 long-format DataFrame
logger.info(f"Extracting per-run ranks for gene prefix '{GENE_NAME}'...")
df_brca2_runs = import_utils.build_single_gene_rank_runs_long_df(
    df_ranks_by_key=df_ranks_by_key,
    gene_name=GENE_NAME,
    dataset_col=DATASET_COL,
    replicate_col=REPLICATE_COL_OUT,
)

# Sanity: expected columns
expected_cols = {DATASET_COL, REPLICATE_COL_OUT, "gene", "gene_name_group", METRIC_COL}
missing_cols = expected_cols - set(df_brca2_runs.columns)
if missing_cols:
    raise RuntimeError(f"build_single_gene_rank_runs_long_df returned DataFrame missing columns: {missing_cols}")

logger.info(f"Per-run BRCA2 table shape: {df_brca2_runs.shape}")

# 3) Sanity checks on pairing & replicates
# ensure replicate column present
if REPLICATE_COL_OUT not in df_brca2_runs.columns:
    raise ValueError(f"Seed column '{REPLICATE_COL_OUT}' not found in BRCA2 runs DataFrame.")

# count replicates per dataset and check uniformity for datasets involved in comparisons
replicate_counts = df_brca2_runs.groupby(DATASET_COL)[REPLICATE_COL_OUT].nunique()
logger.info("Seed counts per dataset (unique replicates):")
for ds, cnt in replicate_counts.items():
    logger.info(f"  {ds}: {cnt}")

# Verify replicate equality for each requested pair
if "dataset_pairs" not in globals() and "dataset_pairs" not in locals():
    raise ValueError("dataset_pairs is not defined. Define dataset_pairs as a list of (A, B) tuples before running.")
for a, b in dataset_pairs:
    if a not in replicate_counts.index:
        raise KeyError(f"Dataset '{a}' not present in BRCA2 runs dataframe.")
    if b not in replicate_counts.index:
        raise KeyError(f"Dataset '{b}' not present in BRCA2 runs dataframe.")

    replicates_a = set(df_brca2_runs.loc[df_brca2_runs[DATASET_COL] == a, REPLICATE_COL_OUT])
    replicates_b = set(df_brca2_runs.loc[df_brca2_runs[DATASET_COL] == b, REPLICATE_COL_OUT])
    if replicates_a != replicates_b:
        # Fail loudly — pairing must be exact
        missing_in_b = sorted(replicates_a - replicates_b)
        missing_in_a = sorted(replicates_b - replicates_a)
        raise ValueError(
            f"Seed mismatch for pair ({a} vs {b}). "
            f"Seeds in {a} but not {b}: {missing_in_b}. "
            f"Seeds in {b} but not {a}: {missing_in_a}."
        )
    logger.info(f"Pair ({a} vs {b}) has {len(replicates_a)} matched replicates.")

# 4) Run paired Wilcoxon using existing util (no rank sign flip)
logger.info("Running paired Wilcoxon signed-rank tests on BRCA2 rank (lower = better)...")
df_brca2_dset_comp = pu.compute_paired_dataset_wilcoxon_results(
    df_runs=df_brca2_runs,
    dataset_pairs=dataset_pairs,
    metric=METRIC_COL,
    replicate_col=REPLICATE_COL_OUT,
    group_cols=[],
    fdr_groupby_cols=None, # adjust if you want FDR grouped by something
    fdr_correction=True,
    require_matching_replicates=True,
)

logger.info("Paired Wilcoxon tests complete. Result table:")
logger.info(f"\n{df_brca2_dset_comp}")

# Return / assign the main object for downstream use
# e.g., assign to a notebook variable
df_brca2_results = df_brca2_dset_comp
logger.info("BRCA2 paired-rank analysis finished.")


#### Suppl Table: BRCA2 rank across different dataset pairs, wilcoxon results

In [ ]:
logging.info("Saving statistical comparisons of BRCA2 rank across different dataset pairs")
df_brca2_results.sort_values("q_fdr_bh").to_csv(
    SAVE_TABLES_DIR
    / f"p1000_datasplit_brca2_rankComparison_{eval_set}Set.csv",
    float_format="%.3f",
    index=False,
)

display(df_brca2_results.sort_values("q_fdr_bh").round(3))

#### Suppl Fig: BRCA2 forest plots

In [ ]:
features = ["somatic", "rare lof", "rare missense"]
baseline_germline = "germline_rare_lof_missense"
baseline_somatic  = "somatic_amp somatic_del somatic_mut"

fig_brca2_forest_germline, _ = plot_utils.plot_forest_vs_baseline_with_feature_glyphs(
    df_brca2_results=df_brca2_results,
    baseline=baseline_germline,
    features=features,
    get_feature_matrix_func=plot_utils.get_feature_matrix_hierarchical,
    title="BRCA2 rank relative to a reference germline dataset",
    glyph_col_spacing=0.70,
    glyph_marker_size=70,
    glyph_w_in=1.1,
)

fig_brca2_forest_germline.canvas.draw()

fig_brca2_forest_germline.savefig(
    os.path.join(
        SAVE_FIGS_DIR, f"datasplit_P1000_brca2_forest_germline_comparisons_{eval_set}_set.png"
    ),
    format="png",
    bbox_inches="tight",
    dpi=600,
)

fig_brca2_forest_somatic, _ = plot_utils.plot_forest_vs_baseline_with_feature_glyphs(
    df_brca2_results=df_brca2_results,
    baseline=baseline_somatic,
    features=features,
    get_feature_matrix_func=plot_utils.get_feature_matrix_hierarchical,
    title="BRCA2 rank relative to somatic-only baseline",
    glyph_col_spacing=0.70,
    glyph_marker_size=70,
    glyph_w_in=1.1,
    )
fig_brca2_forest_somatic.canvas.draw()

fig_brca2_forest_somatic.savefig(
    os.path.join(
        SAVE_FIGS_DIR, f"datasplit_P1000_brca2_forest_somatic_comparisons_{eval_set}_set.png"
    ),
    format="png",
    bbox_inches="tight",
    dpi=600,
)

#### Suppl Fig: BRCA2 per-replicate rank across different datasets

In [ ]:
features = ["somatic", "rare lof", "rare missense"]

somatic_only = "somatic_amp somatic_del somatic_mut"
germline_only = "germline_rare_lof_missense"
somatic_plus_germline = (
    "somatic_amp somatic_del somatic_mut germline_rare_lof_missense"
)

# Germline-only vs somatic-only
fig_BRCA2rank_G_vs_S_per_replicate = plot_utils.plot_paired_gene_rank_with_feature_caption(
    df_brca2_runs,
    a=germline_only,
    b=somatic_only,
    features=features,
    replicate_col=REPLICATE_COL,
    dataset_labels=["Germline only", "Somatic only"],
    title = "Per-split BRCA2 rank\nGermline-only vs. somatic-only"
) 
fig_BRCA2rank_G_vs_S_per_replicate.savefig(
    os.path.join(
        SAVE_FIGS_DIR, f"datasplit_P1000_brca2_G_vs_S_per_replicate_{eval_set}_set.png"
    ),
    format="png",
    bbox_inches="tight",
    dpi=600,
)

# Somatic + germline vs somatic-only
fig_BRCA2rank_SG_vs_S_per_replicate = plot_utils.plot_paired_gene_rank_with_feature_caption(
    df_brca2_runs,
    a=somatic_plus_germline,
    b=somatic_only,
    features=features,
    replicate_col=REPLICATE_COL,
    dataset_labels=["Somatic + germline", "Somatic only"],
    title = "Per-split BRCA2 rank\nSomatic+germline vs. somatic-only"
)

fig_BRCA2rank_SG_vs_S_per_replicate.savefig(
    os.path.join(
        SAVE_FIGS_DIR, f"datasplit_P1000_brca2_SG_vs_S_per_replicate_{eval_set}_set.png"
    ),
    format="png",
    bbox_inches="tight",
    dpi=600,
)

#### Part of Fig 4: BRCA rank per dataset

In [ ]:
logger.info("Any combo of germline only that has rare_lof ranks BRCA2 highly")
df_BRCA2 = import_utils.get_single_gene_info(
    df_imps_by_key, df_ranks_by_key, gene_name="BRCA2", group_identifier_col="datasets"
)
df_BRCA2.sort_values(by=["rank", "absolute_importance"], ascending=[True, False])[
    ["datasets", "gene_name_group", "rank", "absolute_importance", "importance"]
].style.format(
    {"importance": "{:.3f}", "absolute_importance": "{:.3f}", "rank": "{:.0f}"}
)

### non-BRCA2 feature importance results

##### hyperparameters

In [ ]:
# focus on biggest (least sparse germline dataset) tested
dsets_to_visualize = [
    "somatic_amp somatic_del somatic_mut",
    "germline_rare_lof_missense",
    "somatic_amp somatic_del somatic_mut germline_rare_lof_missense",
]

modalities_to_visualize = [
    "somatic_amp",
    "somatic_del",
    "somatic_mut",
    "germline_rare_lof_missense",
]

In [ ]:
logger.info("Processing per-run feature importances and ranks (indexing by replicate)...")
df_imps_by_key, df_ranks_by_key = import_utils.process_feature_importances(
    df_feature_importance_paths=df_filt,
    response_df=response_df,
    importance_path_column=IMPORTANCE_PATH_COL,
    group_identifier_column=GROUP_IDENTIFIER_COL,
    replicate_col=REPLICATE_COL,
)

#### Part of Fig 4: top 10 features per dataset
Component of Figure 4 on top-ranked genes and BRCA2 trajectory.

In [ ]:
top_10_features_by_rank = import_utils.extract_top_features_from_df(
    df_ranks_by_key, top_n=30, keep_smallest_n=True, index_label="rank"
)

# dsets = sorted(top_10_features_by_rank.columns.tolist())[::-1]
display(top_10_features_by_rank[dsets_to_visualize])

#### Suppl Fig 5: top 10 genes plus OR and control_freq information for representative datasets

In [ ]:
# For each of the top features, extract the OR and control frequency from the associated modality.
# E.g., if feature is AR_somatic_mut --> extract OR and control frequency from the somatic_mut DF

logger.info("1. Build modality-level summaries")
input_data_dir = df.input_data_dir.unique()[0]
dset_df_f = [os.path.join(input_data_dir, i + ".csv") for i in modalities_to_visualize]
dset_dfs = [pd.read_csv(p, index_col=0) for p in dset_df_f]

named_modality_dfs = dict(zip(modalities_to_visualize, dset_dfs))
modality_stats = import_utils.build_modality_stats(named_modality_dfs, response_df)

logger.info("2. Annotate each dataset group's top-N table")
logger.info("Load feature ranks by key (gene x modality), not just gene importances; we need to loop back to original dataset to calculate OR and control freq for top features")
feature_importance_path = (
    f"{eval_set}_feature_importances_path"
)
_, df_feat_ranks_by_key = import_utils.process_feature_importances(
    df_filt,
    response_df,
    importance_path_column=feature_importance_path,
    group_identifier_column="datasets",
    replicate_col="split_id",
)

N = 10
top_N_features_by_rank = import_utils.extract_top_features_from_df(
    df_feat_ranks_by_key, top_n=N, keep_smallest_n=True, index_label="rank"
)
annotated_topN = import_utils.annotate_top_features(
    top_N_features_by_rank, dsets_to_visualize, modality_stats
)
for dset in dsets_to_visualize:
    annotated_topN[dset].drop(columns="mu1", inplace=True)
    # annotated_topN[dset].drop(columns='feature', inplace=True)



In [ ]:

logger.info("Display top ranked gene tables with OR and control_freq")
for dset in dsets_to_visualize:
    logger.info(f"\ndset: {dset}")

    # --- rename mu0 -> control_freq (leave all other cols intact) ------------
    df_out = (
        annotated_topN[dset]
        .rename(columns={"mu0": "control_freq"})
        .round({"control_freq": 3, "odds_ratio": 2})
    )

    # --- save to CSV ---------------------------------------------------------
    out_path = os.path.join(
        SAVE_TABLES_DIR, f"{dset}_top_ranked_genes_with_class_freq_and_odds_ratio.csv"
    )
    logger.info(f"Saving to '{out_path}'")
    df_out = df_out.round(3)
    df_out.to_csv(out_path, index_label="rank")

    # --- pretty-print --------------------------------------------------------
    display(
        df_out.style.format(
            {"control_freq": "{:.3f}", "mu1": "{:.3f}", "odds_ratio": "{:.2f}"}
        )
        .background_gradient(subset=["control_freq"], cmap="Blues")
        .background_gradient(subset=["odds_ratio"], cmap="OrRd")
        .set_caption(f"Top ranked gene for data combination: {dset}")
    )
